# 03 assignment

완전한 에이전트 루프를 갖춘 Movie Agent를 구축하세요!

에이전트가 수행해야 할 작업:
- 영화에 대한 사용자 질문을 받습니다.
- 어떤 도구를 호출할지 결정합니다 (필요한 경우).
- 실제로 API를 호출하여 LLM에게 실제 결과를 반환합니다.
- LLM이 최종 답변을 줄 때까지 루프를 계속합니다.

< 요구사항 > 
1. 실제 API를 호출하는 최소 3개의 도구를 갖춘 에이전트를 만드세요:
    - get_popular_movies() - /movies 호출
    - get_movie_details(id) - /movies/:id 호출
    - get_similar_movies(id) - /movies/:id/similar 호출

2. 에이전트가 갖춰야 할 조건:
    - (수동 프롬프팅이 아닌) OpenAI tools 파라미터를 사용하세요.
    - 응답에서 tool_calls를 처리하세요.
    - 실제 API를 호출하고 결과를 다시 전달해야 합니다.
    - 메모리를 활용한 멀티턴 대화를 지원해야 합니다.

3. 예시 상호작용

    User: 지금 인기 있는 영화 알려줘

    Agent: [get_popular_movies() 호출]

    Agent: 현재 인기 영화 목록입니다: 1. 듄: 파트 2 (ID: 693134)...

    
    User: 듄에 대해 더 알려줘

    Agent: [get_movie_details(693134) 호출]

    Agent: 듄: 파트 2는 드니 빌뇌브 감독의 2024년 SF 대작으로...

    User: 비슷한 영화 추천해 줄래?

    Agent: [get_similar_movies(693134) 호출]
    
    Agent: 듄을 좋아하셨다면 이런 영화를 추천드립니다: 블레이드 러너 2049, 어라이벌...

In [1]:
import requests, json

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

def get_popular_movies():
    return requests.get(f"{BASE_URL}/movies").json()

def get_movie_details(id):
    return requests.get(f"{BASE_URL}//movies/{id}").json()

def get_similar_movies(id):
    return requests.get(f"{BASE_URL}//movies/{id}/similar").json()

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기 영화 목록 가져옵니다",
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "id로 영화 상세 정보 가져옵니다",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The id of a movie"
                    }
                }
            },
            "required": ["id"]
        }
    },   
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "id와 유사한 영화를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The id of a movie"
                    }
                }
            },
            "required": ["id"]
        }
    },       
]



In [2]:
import openai
from openai.types.chat import ChatCompletionMessage


client = openai.Client()

messages = []

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role":"assistant",
                "content":message.content or "",
                "tool_calls" : [
                    {
                        "id":tool_call.id,
                        "type":"function",
                        "function":{
                            "name":tool_call.function.name,
                            "arguments":tool_call.function.arguments,
                        }
                    } 
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments            
        
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
                print(f"Error: Invalid JSON arguments for function {function_name}, {arguments}")
                continue

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)
            
            args_str = arguments.get("id", "")
            print(f"Agent: [{function_name}({args_str})] 호출")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )

        call_ai()

    else: 
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}")


def call_ai():
    # print(messages)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    # print(response.choices[0].message)
    process_ai_response(response.choices[0].message)

In [3]:
message = "지금 인기 있는 영화 알려줘"

messages.append(
    {
        "role": "user",
        "content": message
    }
)

print(f"User: {message}")
call_ai()


User: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies()] 호출
Agent: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Obsession**  
   - **개요**: "One Wish Willow"를 깨뜨린 후, 희망 없는 로맨티스트가 사랑을 얻기 위한 소원을 이루게 되지만, 그 소원이 어둡고 음산한 대가를 치르게 됩니다.  
   - **개봉일**: 2026-05-13  
   - **평점**: 7.9  
   - ![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi**  
   - **개요**: 1980년대 안드라프라데시의 시골에서 한 활기찬 마을 사람이 스포츠를 통해 지역사회를 하나로 모아 강력한 적에 대항합니다.  
   - **개봉일**: 2026-06-03  
   - **평점**: 6.45  
   - ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Hai Jawani Toh Ishq Hona Hai**  
   - **개요**: 결혼을 포기한 Jass가 외국에서 새로운 로맨스를 시작하지만 예상치 못한 폭로로 사랑과 충성심의 의미를 되짚어야 합니다.  
   - **개봉일**: 2026-06-04  
   - **평점**: 5.4  
   - ![Hai Jawani Toh Ishq Hona Hai](https://image.tmdb.org/t/p/w780/vmlJvz6qVzYgei2V74GvnmcuQfW.jpg)

4. **The Unknown Man**  
   - **개요**: 플란더스의 작가 Louis가 코트다쥐르에서 고립되며 영감을 얻고자 하는 이야기입니다.  
   - **개봉일**: 2021-10-16  
   - **평점**: 8.06  
   - ![The Unknown Man](htt

In [4]:
message = "Obsession 대해 더 알려줘"

messages.append(
    {
        "role": "user",
        "content": message
    }
)

print(f"User: {message}")
call_ai()


User: Obsession 대해 더 알려줘
Agent: [get_movie_details(1339713)] 호출
Agent: 영화 **Obsession**에 대한 상세 정보는 다음과 같습니다:

- **타이틀**: Obsession
- **개봉일**: 2026-05-13
- **장르**: 공포 (Horror), 스릴러 (Thriller)
- **러닝타임**: 108분
- **평점**: 7.9 / 10 (총 711표)
- **예산**: $750,000
- **수익**: $242,663,965
- **원어**: 영어 (English)
- **태그라인**: "My little food critic."

### 개요
"One Wish Willow"를 깨뜨린 후, 희망 없는 로맨티스트가 사랑을 얻기 위한 소원을 이루게 되지만, 그 소원이 어둡고 음산한 대가를 치르게 됩니다.

### 제작 정보
- **제작사**:
  - Tea Shop Productions (영국)
  - Under the Shell (미국)
  - Capstone Pictures (미국)
  - Blumhouse Productions (미국)

### 공식 홈페이지
[Obsession 공식 홈페이지](http://www.focusfeatures.com/obsession)

### 포스터
![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

이 영화에 대해 더 궁금한 점이 있다면 말씀해 주세요!


In [5]:
message = "비슷한 영화 추천해 줄래?"

messages.append(
    {
        "role": "user",
        "content": message
    }
)

print(f"User: {message}")
call_ai()


User: 비슷한 영화 추천해 줄래?
Agent: [get_similar_movies(1339713)] 호출
Agent: **Obsession**와 비슷한 영화들을 추천합니다:

1. **Unfriended**  
   - **개요**: 한밤 중 화상 채팅 중, 친구들이 자살한 동급생의 메세지를 받으며 벌어지는 이야기를 다룹니다. 무시할 수 없는 비밀들이 드러나고, 그들은 생존을 위해 싸워야 합니다.  
   - **개봉일**: 2014-06-18  
   - **평점**: 5.5  
   - ![Unfriended](https://image.tmdb.org/t/p/w780/8byPxREr3BqBNyyij5WHVSOVtYk.jpg)

2. **Suburban Gothic**  
   - **개요**: 죽은 사람과 대화할 수 있는 실직 남자가 반항적인 바텐더와 팀을 이루어 그들을 괴롭히는 복수의 유령을 찾는 이야기입니다.  
   - **개봉일**: 2014-07-19  
   - **평점**: 5.5  
   - ![Suburban Gothic](https://image.tmdb.org/t/p/w780/y3oWzE9jzCNfJ1B1R1ndVVG5Uh9.jpg)

3. **The Forest of Love**  
   - **개요**: 사기꾼과 영화 제작을 꿈꾸는 팀이 두 청년 여성의 인생에 끼어들지만, 진짜 상황은 달라지고 서로의 비밀이 드러나는 이야기를 다룹니다.  
   - **개봉일**: 2019-10-10  
   - **평점**: 6.3  
   - ![The Forest of Love](https://image.tmdb.org/t/p/w780/A7ve6RmC8C4VmfMEMhwRCwKWeim.jpg)

4. **The Crush**  
   - **개요**: 한 소녀가 naive한 작가에게 집착하게 되고, 그로 인한 심각한 결과를 다루고 있습니다.  
   - **개봉일**: 1993-04-02  
   - **평점**: 6.7  
   - ![Th